## 1. Handling Invalid Zero-Speed Observations

The raw METR-LA dataset contains zero-valued speed observations.

Based on the exploratory analysis performed in `01_data_understanding.ipynb`,
these zero values occur in both short and long consecutive runs. Therefore,
they are treated as missing/invalid sensor observations during preprocessing.

Short gaps will be handled using temporal interpolation, while very long
sensor outages will not be blindly interpolated because doing so could
introduce unrealistic synthetic traffic patterns.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:

file_path = "../data/raw/metr-la.h5"

with h5py.File(file_path, "r") as f:
    sensor_ids = f["df/block0_items"][:]
    timestamps = f["df/axis1"][:]
    values = f["df/block0_values"][:]

sensor_ids = [sid.decode("utf-8") for sid in sensor_ids]

timestamps_readable = pd.to_datetime(timestamps, unit="ns")

traffic_df = pd.DataFrame(
    values,
    index=timestamps_readable,
    columns=sensor_ids
)

print("Dataset loaded successfully!")
print("Shape:", traffic_df.shape)

Dataset loaded successfully!
Shape: (34272, 207)


In [8]:
traffic_clean = traffic_df.copy()

traffic_clean = traffic_clean.replace(0, np.nan)

print("Zero values remaining:", (traffic_clean == 0).sum().sum())
print("Missing values after conversion:", traffic_clean.isnull().sum().sum())

Zero values remaining: 0
Missing values after conversion: 575302


In [9]:

missing_runs = []

for sensor in traffic_clean.columns:
    is_missing = traffic_clean[sensor].isna()

    groups = is_missing.ne(is_missing.shift()).cumsum()
    runs = is_missing.groupby(groups).sum()

    runs = runs[runs > 0]

    missing_runs.extend(runs.tolist())

missing_runs = pd.Series(missing_runs)

print("Total missing runs:", len(missing_runs))

print("\nMissing runs of 1-6 time points:")
print((missing_runs <= 6).sum())

print("\nMissing runs longer than 6 time points:")
print((missing_runs > 6).sum())

print("\nLongest missing run:")
print(int(missing_runs.max()), "time points")

Total missing runs: 25963

Missing runs of 1-6 time points:
16559

Missing runs longer than 6 time points:
9404

Longest missing run:
1683 time points


## 2. Short-Gap Interpolation

Based on the missing-gap analysis, short missing sequences of up to 6
consecutive time points will be handled using linear interpolation.

Since the METR-LA dataset is sampled every 5 minutes, 6 time points
represent a maximum gap of 30 minutes.

Longer missing sequences will remain as missing values because interpolating
large sensor outages may introduce unrealistic traffic patterns.

In [10]:
traffic_interpolated = traffic_clean.interpolate(
    method="linear",
    limit=6,
    limit_area="inside"
)

print("Missing values before interpolation:", traffic_clean.isna().sum().sum())
print("Missing values after interpolation:", traffic_interpolated.isna().sum().sum())

Missing values before interpolation: 575302
Missing values after interpolation: 486911


## 3. Remaining Missing Values

After short-gap interpolation, some missing values remain because they belong
to longer sensor outages.

Before creating model-ready sequences, we will examine how these remaining
missing values are distributed across the sensors.

In [11]:
remaining_missing = traffic_interpolated.isna().sum()

print("Total remaining missing values:", remaining_missing.sum())

print("\nSensors with the highest remaining missing values:")
print(remaining_missing.sort_values(ascending=False).head(10))

Total remaining missing values: 486911

Sensors with the highest remaining missing values:
716939    6038
718076    5911
760987    4931
716955    4907
717510    4483
769405    4409
769418    4409
717513    4156
717576    4054
718204    4043
dtype: int64


## 4. Sensor Missingness Analysis

The remaining missing values are concentrated differently across the
207 sensors.

To determine whether any sensors have excessive unresolved missing data,
we calculate the percentage of missing observations for each sensor.

This analysis will help us decide whether sensor removal is necessary
before creating model-ready sequences.

In [12]:
missing_percentage = (
    traffic_interpolated.isna().sum() / len(traffic_interpolated)
) * 100

print("Average missing percentage across sensors:",
      round(missing_percentage.mean(), 2), "%")

print("\nHighest sensor missing percentages:")
print(missing_percentage.sort_values(ascending=False).head(15))

Average missing percentage across sensors: 6.86 %

Highest sensor missing percentages:
716939    17.617880
718076    17.247316
760987    14.387838
716955    14.317810
717510    13.080649
769405    12.864729
769418    12.864729
717513    12.126517
717576    11.828898
718204    11.796802
717490    10.825163
717491    10.819328
769430    10.696779
767454    10.177404
773869     9.774743
dtype: float64


In [13]:

print("Sensors with > 5% missing values:",
      (missing_percentage > 5).sum())

print("Sensors with > 10% missing values:",
      (missing_percentage > 10).sum())

print("Sensors with > 15% missing values:",
      (missing_percentage > 15).sum())

print("Sensors with > 20% missing values:",
      (missing_percentage > 20).sum())

Sensors with > 5% missing values: 207
Sensors with > 10% missing values: 14
Sensors with > 15% missing values: 2
Sensors with > 20% missing values: 0


In [14]:

n = len(traffic_interpolated)

train_end = int(n * 0.70)
val_end = int(n * 0.80)

train_data = traffic_interpolated.iloc[:train_end].copy()
val_data = traffic_interpolated.iloc[train_end:val_end].copy()
test_data = traffic_interpolated.iloc[val_end:].copy()

print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)
print("Test data:", test_data.shape)

print("\nDate ranges:")
print("Train:", train_data.index.min(), "to", train_data.index.max())
print("Validation:", val_data.index.min(), "to", val_data.index.max())
print("Test:", test_data.index.min(), "to", test_data.index.max())

Training data: (23990, 207)
Validation data: (3427, 207)
Test data: (6855, 207)

Date ranges:
Train: 2012-03-01 00:00:00 to 2012-05-23 07:05:00
Validation: 2012-05-23 07:10:00 to 2012-06-04 04:40:00
Test: 2012-06-04 04:45:00 to 2012-06-27 23:55:00


In [39]:

target_start = INPUT_STEPS + FORECAST_HORIZON - 1

train_target_mask = train_data.iloc[target_start:].notna().to_numpy()
val_target_mask = val_data.iloc[target_start:].notna().to_numpy()
test_target_mask = test_data.iloc[target_start:].notna().to_numpy()

print("Train target mask:", train_target_mask.shape)
print("Validation target mask:", val_target_mask.shape)
print("Test target mask:", test_target_mask.shape)

print(
    "Validation observed target percentage:",
    round(100 * val_target_mask.mean(), 2),
    "%"
)

Train target mask: (23973, 207)
Validation target mask: (3410, 207)
Test target mask: (6838, 207)
Validation observed target percentage: 94.19 %


In [40]:

np.save(
    "../data/processed/train_target_mask.npy",
    train_target_mask
)

np.save(
    "../data/processed/validation_target_mask.npy",
    val_target_mask
)

np.save(
    "../data/processed/test_target_mask.npy",
    test_target_mask
)

print("Target validity masks saved successfully.")

Target validity masks saved successfully.


In [15]:

train_missing = train_data.isna().sum().sum()
val_missing = val_data.isna().sum().sum()
test_missing = test_data.isna().sum().sum()

print("Remaining missing values:")
print("Training:", train_missing)
print("Validation:", val_missing)
print("Test:", test_missing)

Remaining missing values:
Training: 296992
Validation: 41046
Test: 148873


In [16]:

train_missing_pct = (train_missing / train_data.size) * 100
val_missing_pct = (val_missing / val_data.size) * 100
test_missing_pct = (test_missing / test_data.size) * 100

print("Missing-value percentage:")
print("Training:", round(train_missing_pct, 2), "%")
print("Validation:", round(val_missing_pct, 2), "%")
print("Test:", round(test_missing_pct, 2), "%")

Missing-value percentage:
Training: 5.98 %
Validation: 5.79 %
Test: 10.49 %


In [17]:

train_complete = train_data.notna().all(axis=1)
val_complete = val_data.notna().all(axis=1)
test_complete = test_data.notna().all(axis=1)

print("Complete timestamps:")
print("Training:", train_complete.sum(), "out of", len(train_data))
print("Validation:", val_complete.sum(), "out of", len(val_data))
print("Test:", test_complete.sum(), "out of", len(test_data))

Complete timestamps:
Training: 5876 out of 23990
Validation: 909 out of 3427
Test: 500 out of 6855


In [18]:
print("Percentage of complete timestamps:")
print("Training:", round(train_complete.mean() * 100, 2), "%")
print("Validation:", round(val_complete.mean() * 100, 2), "%")
print("Test:", round(test_complete.mean() * 100, 2), "%")

Percentage of complete timestamps:
Training: 24.49 %
Validation: 26.52 %
Test: 7.29 %


In [19]:

sensor_medians = train_data.median()

print("Training sensor medians calculated.")
print("\nFirst 10 sensor medians:")
print(sensor_medians.head(10))

Training sensor medians calculated.

First 10 sensor medians:
773869    65.500000
767541    65.125000
767542    67.444444
717447    54.333333
717446    48.125000
717445    57.222222
773062    62.750000
767620    63.625000
737529    62.750000
717816    66.250000
dtype: float64


In [20]:

train_model_input = train_data.copy()
val_model_input = val_data.copy()
test_model_input = test_data.copy()


train_model_input = train_model_input.fillna(sensor_medians)
val_model_input = val_model_input.fillna(sensor_medians)
test_model_input = test_model_input.fillna(sensor_medians)

print("Remaining missing values after model-input imputation:")
print("Training:", train_model_input.isna().sum().sum())
print("Validation:", val_model_input.isna().sum().sum())
print("Test:", test_model_input.isna().sum().sum())

Remaining missing values after model-input imputation:
Training: 0
Validation: 0
Test: 0


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(train_model_input)

print("Scaler fitted using training data only.")

Scaler fitted using training data only.


In [22]:

train_scaled = pd.DataFrame(
    scaler.transform(train_model_input),
    index=train_model_input.index,
    columns=train_model_input.columns
)

val_scaled = pd.DataFrame(
    scaler.transform(val_model_input),
    index=val_model_input.index,
    columns=val_model_input.columns
)

test_scaled = pd.DataFrame(
    scaler.transform(test_model_input),
    index=test_model_input.index,
    columns=test_model_input.columns
)

print("Training scaled shape:", train_scaled.shape)
print("Validation scaled shape:", val_scaled.shape)
print("Test scaled shape:", test_scaled.shape)

Training scaled shape: (23990, 207)
Validation scaled shape: (3427, 207)
Test scaled shape: (6855, 207)


In [23]:
print("Average training mean:",
      round(train_scaled.mean().mean(), 6))

print("Average training standard deviation:",
      round(train_scaled.std().mean(), 6))

Average training mean: 0.0
Average training standard deviation: 1.000021


In [24]:
print("Remaining missing values:")
print("Training:", train_scaled.isna().sum().sum())
print("Validation:", val_scaled.isna().sum().sum())
print("Test:", test_scaled.isna().sum().sum())

Remaining missing values:
Training: 0
Validation: 0
Test: 0


In [25]:

INPUT_STEPS = 12       
FORECAST_HORIZON = 6   

print("Input window:", INPUT_STEPS, "time steps")
print("Forecast horizon:", FORECAST_HORIZON, "time steps")
print("Sampling interval: 5 minutes")
print("Historical window:", INPUT_STEPS * 5, "minutes")
print("Forecast horizon:", FORECAST_HORIZON * 5, "minutes")

Input window: 12 time steps
Forecast horizon: 6 time steps
Sampling interval: 5 minutes
Historical window: 60 minutes
Forecast horizon: 30 minutes


In [26]:
def create_sequences(data, input_steps, forecast_horizon):
    """
    Create time-series input/target sequences.

    X contains the previous `input_steps` observations.
    y contains the traffic state `forecast_horizon` steps ahead.
    """

    values = data.to_numpy(dtype=np.float32)

    X = []
    y = []

    for i in range(input_steps, len(values) - forecast_horizon + 1):
        X.append(values[i - input_steps:i])
        y.append(values[i + forecast_horizon - 1])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    return X, y

In [27]:
X_train, y_train = create_sequences(
    train_scaled,
    INPUT_STEPS,
    FORECAST_HORIZON
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (23973, 12, 207)
y_train shape: (23973, 207)


In [28]:
X_val, y_val = create_sequences(
    val_scaled,
    INPUT_STEPS,
    FORECAST_HORIZON
)

X_test, y_test = create_sequences(
    test_scaled,
    INPUT_STEPS,
    FORECAST_HORIZON
)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_val shape: (3410, 12, 207)
y_val shape: (3410, 207)

X_test shape: (6838, 12, 207)
y_test shape: (6838, 207)


In [29]:

first_input_start = train_scaled.index[0]
first_input_end = train_scaled.index[INPUT_STEPS - 1]
first_target_time = train_scaled.index[INPUT_STEPS + FORECAST_HORIZON - 1]

print("First input starts:", first_input_start)
print("Last input observation:", first_input_end)
print("Target timestamp:", first_target_time)

print(
    "\nTime between last input and target:",
    first_target_time - first_input_end
)

First input starts: 2012-03-01 00:00:00
Last input observation: 2012-03-01 00:55:00
Target timestamp: 2012-03-01 01:25:00

Time between last input and target: 0 days 00:30:00


In [33]:
import os

path = "../data/processed"

print("Exists:", os.path.exists(path))
print("Is directory:", os.path.isdir(path))
print("Is file:", os.path.isfile(path))

Exists: True
Is directory: False
Is file: True


In [34]:
os.remove("../data/processed")
os.makedirs("../data/processed", exist_ok=True)

print("data/processed directory created successfully.")

data/processed directory created successfully.


In [35]:
import os
import json
import joblib

os.makedirs("../data/processed", exist_ok=True)

np.savez_compressed(
    "../data/processed/train_sequences.npz",
    X=X_train,
    y=y_train
)

np.savez_compressed(
    "../data/processed/validation_sequences.npz",
    X=X_val,
    y=y_val
)

np.savez_compressed(
    "../data/processed/test_sequences.npz",
    X=X_test,
    y=y_test
)

joblib.dump(
    scaler,
    "../data/processed/traffic_scaler.pkl"
)

print("Processed sequences and scaler saved successfully.")

Processed sequences and scaler saved successfully.


In [36]:
import os

print("\nSaved files:")
for filename in sorted(os.listdir("../data/processed")):
    print("-", filename)


Saved files:
- test_sequences.npz
- traffic_scaler.pkl
- train_sequences.npz
- validation_sequences.npz


In [37]:
metadata = {
    "dataset": "METR-LA",
    "num_sensors": 207,
    "sampling_interval_minutes": 5,
    "input_steps": INPUT_STEPS,
    "historical_window_minutes": INPUT_STEPS * 5,
    "forecast_horizon_steps": FORECAST_HORIZON,
    "forecast_horizon_minutes": FORECAST_HORIZON * 5,
    "train_split": 0.70,
    "validation_split": 0.10,
    "test_split": 0.20,
    "train_sequences": int(len(X_train)),
    "validation_sequences": int(len(X_val)),
    "test_sequences": int(len(X_test)),
    "sensor_ids": traffic_df.columns.tolist()
}

with open("../data/processed/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved successfully.")

Metadata saved successfully.


In [38]:
print("Processed data directory:")
for filename in sorted(os.listdir("../data/processed")):
    print("-", filename)

Processed data directory:
- metadata.json
- test_sequences.npz
- traffic_scaler.pkl
- train_sequences.npz
- validation_sequences.npz
